In [7]:
import os
from pathlib import Path

zip_file = "/home/philip/Downloads/pistol-granada.v1-1.yolov8.zip"

if not os.path.exists(zip_file):
    print(f" Файл {zip_file} не найден!")
    print("Положи ZIP-архив с датасетом в папку с ноутбуком")
    print("Или укажи правильный путь в переменной zip_file")
else:
    print(f" Найден: {zip_file}")

 Найден: /home/philip/Downloads/pistol-granada.v1-1.yolov8.zip


In [8]:
import sys
print(sys.executable)

/home/philip/Downloads/myenv/bin/python


In [9]:
# ============ ЯЧЕЙКА 2: РАСПАКОВКА ============
import zipfile
import os

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('dataset')
    print(" Архив распакован!")

print("\n Содержимое:")
for item in os.listdir('dataset'):
    print(f"  {item}")

 Архив распакован!

 Содержимое:
  data.yaml
  test
  README.roboflow.txt
  README.dataset.txt
  train


In [10]:
import yaml
from pathlib import Path

yaml_files = list(Path('dataset').rglob('data.yaml'))
if yaml_files:
    yaml_path = yaml_files[0]
    print(f" data.yaml найден: {yaml_path}")

    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
        print("\nКонфигурация:")
        print(f" Классы: {config.get('names')}")
        print(f" Количество классов: {config.get('nc')}")
else:
    print("data.yaml не найден!")
    print("Ищем структуру...")
    for root, dirs, files in os.walk('dataset'):
        print(f"  {root}")
        if files[:3]:
            print(f" Файлы: {files[:3]}")

 data.yaml найден: dataset/data.yaml

Конфигурация:
 Классы: ['pistol']
 Количество классов: 1


In [11]:
dataset_path = "dataset"

for split in ['train', 'valid', 'test']:
    img_path = Path(f'{dataset_path}/{split}/images')
    lbl_path = Path(f'{dataset_path}/{split}/labels')

    if img_path.exists():
        img_count = len(list(img_path.glob('*')))
        lbl_count = len(list(lbl_path.glob('*'))) if lbl_path.exists() else 0
        print(f"{split}: {img_count} изображений, {lbl_count} аннотаций")
    else:
        alt_path = Path(f'{dataset_path}/{split}')
        if alt_path.exists():
            print(f" {split}: найдена папка {alt_path}")
            print(f"   Содержимое: {list(alt_path.glob('*'))[:5]}")

train: 2687 изображений, 2687 аннотаций
test: 299 изображений, 299 аннотаций


In [12]:
#СОЗДАНИЕ валидационной выборки (тк нет в изначальном датасете) 
import shutil
from sklearn.model_selection import train_test_split

os.makedirs('dataset/valid/images', exist_ok=True)
os.makedirs('dataset/valid/labels', exist_ok=True)

train_images = list(Path('dataset/train/images').glob('*'))
print(f"Найдено {len(train_images)} изображений в train")

if len(train_images) > 10:
    train_files, val_files = train_test_split(
        train_images,
        test_size=0.1,
        random_state=42
    )

    print(f"Train: {len(train_files)} изображений")
    print(f"Valid: {len(val_files)} изображений")

    for img_path in val_files:
        shutil.copy2(img_path, f'dataset/valid/images/{img_path.name}')
        label_path = Path(f'dataset/train/labels/{img_path.stem}.txt')
        if label_path.exists():
            shutil.copy2(label_path, f'dataset/valid/labels/{label_path.name}')

    print(" Valid создан!")
else:
    print("Мало изображений, valid не создается")

Найдено 2687 изображений в train
Train: 2418 изображений
Valid: 269 изображений
 Valid создан!


In [13]:
with open('dataset/data.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['val'] = 'valid/images'

with open('dataset/data.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("data.yaml обновлен:")

for split in ['train', 'valid', 'test']:
    img_path = Path(f'dataset/{split}/images')
    lbl_path = Path(f'dataset/{split}/labels')
    if img_path.exists():
        img_count = len(list(img_path.glob('*')))
        lbl_count = len(list(lbl_path.glob('*'))) if lbl_path.exists() else 0
        print(f"  {split}: {img_count} изображений, {lbl_count} аннотаций")

data.yaml обновлен:
  train: 2687 изображений, 2687 аннотаций
  valid: 269 изображений, 269 аннотаций
  test: 299 изображений, 299 аннотаций


In [8]:
import torch
import os

os.environ.pop('CUDA_VISIBLE_DEVICES', None)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print(f"CUDA_VISIBLE_DEVICES: {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"CUDA доступна: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    device = 0
    batch = 4 # 16 
else:
    print(" GPU не обнаружен")
    device = "cpu"
    batch = 4

from ultralytics import YOLO

print(f"\nОБУЧЕНИЕ НА {str(device).upper()}...")

model = YOLO("yolo26n.pt")
model.train(
    data="dataset/data.yaml",
    epochs=1,  # Для теста — 1 эпоха / 80 для обучения
    imgsz=416,
    batch=batch,
    device=device,
    workers=2 if device == 0 else 2, # workers = 4 
    name="t4_test"
)

print("Тест завершен")

CUDA_VISIBLE_DEVICES: 0
CUDA доступна: True
  GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU

🚀 ОБУЧЕНИЕ НА 0...
Ultralytics 8.4.84 🚀 Python-3.12.3 torch-2.12.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 3769MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, 

In [14]:
import shutil
from pathlib import Path
import os

os.makedirs("pistol_report", exist_ok=True)

files_to_copy = [
    ("dataset/data.yaml", "pistol_report/data.yaml"),
    ("runs/detect/t4_test/weights/best.pt", "pistol_report/best.pt"),
    ("runs/detect/t4_test/results.png", "pistol_report/results.png"),
    ("runs/detect/t4_test/confusion_matrix.png", "pistol_report/confusion_matrix.png"),
]

for src, dst in files_to_copy:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"{dst}")
    else:
        print(f"{src} не найден")

print("\nВсе файлы собраны в pistol_report/")

pistol_report/data.yaml
runs/detect/t4_test/weights/best.pt не найден
runs/detect/t4_test/results.png не найден
runs/detect/t4_test/confusion_matrix.png не найден

Все файлы собраны в pistol_report/


In [24]:
import shutil
from pathlib import Path
import os
from ultralytics import YOLO
# 1. Берем 5 изображений из test набора
os.makedirs("test_images", exist_ok=True)

test_images = list(Path("dataset/test/images").glob("*"))[:5]

for i, img_path in enumerate(test_images, 1):
    shutil.copy(img_path, f"test_images/test_{i}_{img_path.name}")
    print(f"  {i}. {img_path.name}")

print(f"\n5 изображений готовы в папке test_images/")

# 2. Делаем инференс
from ultralytics import YOLO

model = YOLO("/home/philip/Downloads/Telegram Desktop/Новая сжатая ZIP-папка/train-2/weights/best.pt")

results = model.predict(
    source="banana",
    conf=0.25,
    save=True,
    project="test_results",
    name="inference"
)

print("\nИнференс завершен!")

# 3. Показываем результаты
from IPython.display import Image, display
import glob

result_images = glob.glob("test_results/inference/*.jpg")
for img_path in result_images[:3]:
    display(Image(img_path, width=300))

  1. armas-1-_jpg.rf.0efc9982f15fd44a45aa10033a2efcce.jpg
  2. armas-1776-_jpg.rf.af38dce941bcc105592b4ba0d2ddaeb7.jpg
  3. armas-1300-_jpg.rf.4271d79b6aaab399d40767e0b46a29a5.jpg
  4. armas-2591-_jpg.rf.7ecb8482191f9dfb0d5932fe25e8afa9.jpg
  5. armas-2555-_jpg.rf.fa74ce536091e22ec25ad53f54603886.jpg

5 изображений готовы в папке test_images/

image 1/5 /home/philip/Desktop/banana/Pasted image (2).png: 448x640 (no detections), 85.5ms
image 2/5 /home/philip/Desktop/banana/Pasted image (3).png: 448x640 1 pistol, 10.7ms
image 3/5 /home/philip/Desktop/banana/Pasted image (4).png: 448x640 (no detections), 11.3ms
image 4/5 /home/philip/Desktop/banana/Pasted image (5).png: 448x640 1 pistol, 12.4ms
image 5/5 /home/philip/Desktop/banana/Pasted image.png: 448x640 (no detections), 12.7ms
Speed: 2.3ms preprocess, 26.5ms inference, 0.5ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /home/philip/Desktop/runs/detect/test_results/inference-3

Инференс завершен!


In [5]:
import sys
print(sys.executable)

/home/philip/Downloads/myenv/bin/python


In [25]:
from ultralytics import YOLO
import os
import shutil
from pathlib import Path

model = YOLO("/home/philip/Downloads/Telegram Desktop/Новая сжатая ZIP-папка/train-2/weights/best.pt")
metrics = model.val(data="dataset/data.yaml", imgsz=416)

metrics_text = f"""
МЕТРИКИ МОДЕЛИ (Pistol Detection)
=====================================
Precision:  {metrics.box.mp:.4f}
Recall:     {metrics.box.mr:.4f}
mAP50:      {metrics.box.map50:.4f}
mAP50-95:   {metrics.box.map:.4f}
mAP75:      {metrics.box.map75:.4f}
=====================================
"""

with open("pistol_report/metrics.txt", "w") as f:
    f.write(metrics_text)

print("metrics.txt создан!")

# Копируем результаты инференса
print("\n2. Копируем результаты инференса...")

result_dirs = [
    "runs/detect/test_results/inference",
    "runs/detect/predict",
    "test_results/inference"
]

found = False
for dir_path in result_dirs:
    if os.path.exists(dir_path):
        images = list(Path(dir_path).glob("*.jpg"))
        if images:
            os.makedirs("pistol_report/inference_results", exist_ok=True)
            for img in images:
                shutil.copy(img, f"pistol_report/inference_results/{img.name}")
            print(f"Найдено и скопировано {len(images)} изображений из {dir_path}")
            found = True
            break

if not found:
    print(" Результаты инференса не найдены!")

print("\n3. Создаем архив...")
import zipfile

with zipfile.ZipFile('final_submit.zip', 'w') as zipf:
    for root, dirs, files in os.walk('pistol_report'):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.join('pistol_report', os.path.relpath(file_path, 'pistol_report'))
            zipf.write(file_path, arcname)

print(" Архив создан: final_submit.zip")
print("\nВ АРХИВЕ:")
for item in Path("pistol_report").rglob("*"):
    if item.is_file():
        size = item.stat().st_size
        if size > 1024*1024:
            size_str = f"{size/1024/1024:.2f} MB"
        else:
            size_str = f"{size/1024:.1f} KB"
        print(f"   {item.name} ({size_str})")

Ultralytics 8.4.89 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 3769MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1256.0±1309.5 MB/s, size: 22.3 KB)
val: Scanning /home/philip/Desktop/dataset/valid/labels.cache... 269 images, 37 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 269/269 86.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 10.3it/s 1.6s0.1s
                   all        269        266      0.918      0.886      0.959      0.798
Speed: 1.1ms preprocess, 2.2ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /home/philip/Desktop/runs/detect/val-6
metrics.txt создан!

2. Копируем результаты инференса...
Найдено и скопировано 5 изображений из runs/detect/test_results/inference

3. Создаем архив...
 Архив создан: final_submit.zip

В АРХИВЕ:
   data.yaml (0.3 KB)
  

In [27]:

from IPython.display import Image, display
import os
from pathlib import Path
import glob

train_dirs = sorted(Path("runs/detect").glob("t4_test*"))
if train_dirs:
    latest_dir = train_dirs[-1]  # Берём последнюю
    print(f"Результаты из: {latest_dir}")
    print("\n" + "="*50)
    print("ГРАФИКИ ОБУЧЕНИЯ")
    print("="*50)
    
    # results.png 
    results_path = latest_dir / "results.png"
    if results_path.exists():
        print("\n results.png — динамика ошибок и метрик по эпохам")
        display(Image(filename=str(results_path), width=700))
    else:
        print(" results.png не найден")
    
    # confusion_matrix.png — матрица ошибок
    cm_path = latest_dir / "confusion_matrix.png"
    if cm_path.exists():
        print("\n confusion_matrix.png — ошибки между классами")
        display(Image(filename=str(cm_path), width=500))
    else:
        print(" confusion_matrix.png не найден")
    
    # labels.jpg 
    labels_path = latest_dir / "labels.jpg"
    if labels_path.exists():
        print("\nlabels.jpg — примеры размеченных объектов")
        display(Image(filename=str(labels_path), width=700))
    else:
        print(" labels.jpg не найден")
    
   
    print("\n" + "="*50)
    print(" ПРИМЕРЫ БАТЧЕЙ")
    print("="*50)
    
    train_batches = sorted(latest_dir.glob("train_batch*.jpg"))
    if train_batches:
        print(f"\n Найдено {len(train_batches)} батчей обучения, показываем первые 2:")
        for i, img_path in enumerate(train_batches[:2], 1):
            print(f"\nБатч обучения {i}:")
            display(Image(filename=str(img_path), width=600))
    else:
        print(" train_batch*.jpg не найдены")

    print("\n" + "="*50)
    print(" РЕЗУЛЬТАТЫ ВАЛИДАЦИИ")
    print("="*50)
    
    val_batches = sorted(latest_dir.glob("val_batch*_pred.jpg"))
    if val_batches:
        print(f"\n Найдено {len(val_batches)} батчей валидации, показываем первые 2:")
        for i, img_path in enumerate(val_batches[:2], 1):
            print(f"\nПредсказания на валидации {i}:")
            display(Image(filename=str(img_path), width=600))
    else:
        print("val_batch*_pred.jpg не найдены")
 
    print("\n" + "="*50)
    print(" ДЕТАЛЬНЫЕ МЕТРИКИ (из results.csv)")
    print("="*50)
    
    csv_path = latest_dir / "results.csv"
    if csv_path.exists():
        import pandas as pd
        df = pd.read_csv(csv_path)
        print("\n Первые 5 строк метрик:")
        display(df.head())

        import matplotlib.pyplot as plt
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # График box_loss
        if 'train/box_loss' in df.columns and 'val/box_loss' in df.columns:
            axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='blue')
            axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='red')
            axes[0, 0].set_title('Box Loss')
            axes[0, 0].set_xlabel('Epoch')
            axes[0, 0].legend()
            axes[0, 0].grid(True)
        
        # График cls_loss
        if 'train/cls_loss' in df.columns and 'val/cls_loss' in df.columns:
            axes[0, 1].plot(df['epoch'], df['train/cls_loss'], label='Train Cls Loss', color='blue')
            axes[0, 1].plot(df['epoch'], df['val/cls_loss'], label='Val Cls Loss', color='red')
            axes[0, 1].set_title('Classification Loss')
            axes[0, 1].set_xlabel('Epoch')
            axes[0, 1].legend()
            axes[0, 1].grid(True)
        
        # График mAP50
        if 'metrics/mAP50' in df.columns:
            axes[1, 0].plot(df['epoch'], df['metrics/mAP50'], label='mAP50', color='green')
            axes[1, 0].set_title('mAP50')
            axes[1, 0].set_xlabel('Epoch')
            axes[1, 0].legend()
            axes[1, 0].grid(True)
        
        # График mAP50-95
        if 'metrics/mAP50-95' in df.columns:
            axes[1, 1].plot(df['epoch'], df['metrics/mAP50-95'], label='mAP50-95', color='orange')
            axes[1, 1].set_title('mAP50-95 (строгая метрика)')
            axes[1, 1].set_xlabel('Epoch')
            axes[1, 1].legend()
            axes[1, 1].grid(True)
        
        plt.tight_layout()
        plt.show()
    else:
        print("results.csv не найден (его нет при 1 эпохе)")
    
    print("\nВсе графики показаны!")
else:
    print("Папка с результатами не найдена. необходимо запустить обучение.")

Папка с результатами не найдена. необходимо запустить обучение.


In [17]:
os.makedirs("pistol_report", exist_ok=True)

train_dirs = sorted(Path("runs/detect").glob("t4_test*"))
if train_dirs:
    latest_dir = train_dirs[-1]
    print(f"Копируем из: {latest_dir}")
    
    files_to_copy = [
        ("dataset/data.yaml", "pistol_report/data.yaml"),
        (f"{latest_dir}/weights/best.pt", "pistol_report/best.pt"),
        (f"{latest_dir}/results.png", "pistol_report/results.png"),
        (f"{latest_dir}/confusion_matrix.png", "pistol_report/confusion_matrix.png"),
        (f"{latest_dir}/labels.jpg", "pistol_report/labels.jpg"),
        (f"{latest_dir}/results.csv", "pistol_report/results.csv"),
    ]
    
    # Копируем основные файлы
    for src, dst in files_to_copy:
        if os.path.exists(src):
            shutil.copy(src, dst)
            print(f"{dst}")
        else:
            print(f"{src} не найден (пропускаем)")
    
    # Копируем все train_batch*.jpg
    train_batches = list(latest_dir.glob("train_batch*.jpg"))
    if train_batches:
        os.makedirs("pistol_report/train_batches", exist_ok=True)
        for img in train_batches:
            shutil.copy(img, f"pistol_report/train_batches/{img.name}")
        print(f" Скопировано {len(train_batches)} train_batch изображений")
    
    # Копируем все val_batch*_pred.jpg
    val_batches = list(latest_dir.glob("val_batch*_pred.jpg"))
    if val_batches:
        os.makedirs("pistol_report/val_batches", exist_ok=True)
        for img in val_batches:
            shutil.copy(img, f"pistol_report/val_batches/{img.name}")
        print(f" Скопировано {len(val_batches)} val_batch изображений")
    
    print("\n Все файлы собраны в pistol_report/")
else:
    print("Папка с обучением не найдена")

Копируем из: runs/detect/t4_test-3
pistol_report/data.yaml
pistol_report/best.pt
pistol_report/results.png
pistol_report/confusion_matrix.png
pistol_report/labels.jpg
pistol_report/results.csv
 Скопировано 3 train_batch изображений
 Скопировано 3 val_batch изображений

 Все файлы собраны в pistol_report/
